## ℹ️ Sobre este Notebook

Este notebook apresenta uma **análise prática de dados reais de criptomoedas** (BTC, ETH, ADA). Explora dados OHLCV (Open, High, Low, Close, Volume), calcula indicadores técnicos (RSI, Médias Móveis) e prepara os dados para pipelines de Machine Learning.


# Exemplo 12: Análise de Dados Reais de Criptomoedas

**Objetivos:**
- Carregar dados reais de criptomoedas (BTC, ETH, ADA)
- Explorar dados OHLCV (Open, High, Low, Close, Volume)
- Calcular indicadores técnicos (RSI, Médias Móveis)
- Preparar dados para Machine Learning

## 📋 Setup Inicial

### Opção 1: Google Colab
```python
# Upload dos ficheiros .parquet para o Colab
from google.colab import files
uploaded = files.upload()  # Selecionar os ficheiros da pasta data/
```

### Opção 2: Ambiente Local (recomendado)
```bash
# Clonar o repositório
git clone https://github.com/pesobreiro/big_data.git
cd big_data
jupyter lab
```

Os dados estão na pasta `data/` e os notebooks em `notebooks/`.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lag, avg, max, min, stddev
from pyspark.sql.window import Window
import matplotlib.pyplot as plt

spark = SparkSession.builder \
    .appName("Crypto_Analysis") \
    .config("spark.sql.ansi.enabled", "false")\
    .getOrCreate()

print(f"Spark version: {spark.version}")

## 1. Carregar Dados Reais

Dados da Binance (formato Parquet) com OHLCV.

In [ ]:
# Carregar dados BTC ( timeframe 4 horas )
btc_4h = spark.read.parquet("../data/btc_04h_usdt_binance.parquet")

print("Schema BTC (4h):")
btc_4h.printSchema()
print(f"\nTotal de registos: {btc_4h.count()}")
btc_4h.show(5)

In [ ]:
# Carregar múltiplos ativos (exemplo para Tema 1 - Multi-Asset)
assets = ["btc", "eth", "ada"]  # dados disponíveis em data/
dfs = {}

for asset in assets:
    try:
        df = spark.read.parquet(f"../data/{asset}_04h_usdt_binance.parquet")
        df = df.withColumn("asset", col("symbol"))
        dfs[asset] = df
        print(f"{asset.upper()}: {df.count()} registos")
    except Exception as e:
        print(f"Erro ao carregar {asset}: {e}")

# Unir todos num DataFrame
from functools import reduce
from pyspark.sql import DataFrame

def unionAll(dfs):
    return reduce(DataFrame.unionByName, dfs)

all_assets = unionAll(list(dfs.values()))
print(f"\nTotal combinado: {all_assets.count()} registos")

## 2. Indicadores Técnicos (Tema 1)

Calcular Médias Móveis e RSI.

In [ ]:
# Médias Móveis Simples (SMA)
window_spec_7 = Window.partitionBy("symbol").orderBy("open_time").rowsBetween(-6, 0)
window_spec_30 = Window.partitionBy("symbol").orderBy("open_time").rowsBetween(-29, 0)

btc_with_ma = btc_4h \
    .withColumn("sma_7", avg("close").over(window_spec_7)) \
    .withColumn("sma_30", avg("close").over(window_spec_30))

btc_with_ma.select("open_time", "close", "sma_7", "sma_30").show(10)

In [ ]:
# Calcular Retornos e RSI (simplificado)
window_lag = Window.partitionBy("symbol").orderBy("open_time")

btc_returns = btc_4h \
    .withColumn("close_d", col("close").cast("double")) \
    .withColumn("prev_close", lag("close_d", 1).over(window_lag)) \
    .withColumn("price_return", (col("close_d") - col("prev_close")) / col("prev_close")) \
    .na.drop(subset=["price_return"])

btc_returns.select("open_time", "close_d", "price_return").show(10)

# Estatísticas dos retornos
from pyspark.sql.functions import mean as spark_mean, stddev as spark_std
stats = btc_returns.select(
    spark_mean("price_return").alias("mean_return"),
    spark_std("price_return").alias("volatility")
).collect()[0]

mean_r = stats.mean_return
vol = stats.volatility
print(f"\nMédia dos retornos diários: {mean_r:.6f}" if mean_r is not None else "N/A")
print(f"Volatilidade (desvio padrão): {vol:.6f}" if vol is not None else "N/A")

## 3. Feature Engineering para ML

In [ ]:
# Criar features para modelo de classificação (Tema 2 - Regimes de Mercado)
window_7 = Window.partitionBy("symbol").orderBy("open_time").rowsBetween(-6, 0)
window_14 = Window.partitionBy("symbol").orderBy("open_time").rowsBetween(-13, 0)

features_df = btc_4h \
    .withColumn("returns", (col("close") - lag("close", 1).over(window_lag)) / lag("close", 1).over(window_lag)) \
    .withColumn("volatility_7d", stddev("returns").over(window_7)) \
    .withColumn("volume_sma_7", avg("volume").over(window_7)) \
    .withColumn("price_range", (col("high") - col("low")) / col("open")) \
    .na.drop()

# Selecionar features
feature_cols = ["returns", "volatility_7d", "volume_sma_7", "price_range"]
ml_df = features_df.select("open_time", "close", *feature_cols)

print("Features criadas:")
ml_df.show(10)

print(f"\nDataset final para ML: {ml_df.count()} registos")

## 4. Guardar Dados Processados (Bronze → Silver → Gold)

In [ ]:
# Silver: dados limpos com indicadores
btc_with_ma.write \
    .mode("overwrite") \
    .parquet("data/silver/btc_with_indicators.parquet")

# Gold: dataset ML-ready
ml_df.write \
    .mode("overwrite") \
    .parquet("data/gold/btc_ml_ready.parquet")

print("Dados guardados em arquitetura Medallion!")
print("  - Bronze: dados originais")
print("  - Silver: dados limpos + indicadores")
print("  - Gold: dataset para ML")

## 5. Visualização (conversão para Pandas)

In [ ]:
# Converter amostra para Pandas (apenas para visualização!)
pdf = btc_with_ma.select("open_time", "close", "sma_7", "sma_30").toPandas()

plt.figure(figsize=(12, 6))
plt.plot(pdf["open_time"], pdf["close"], label="Preço Close", alpha=0.7)
plt.plot(pdf["open_time"], pdf["sma_7"], label="SMA 7", linestyle="--")
plt.plot(pdf["open_time"], pdf["sma_30"], label="SMA 30", linestyle="--")
plt.title("BTC/USD - Preço e Médias Móveis")
plt.xlabel("Data")
plt.ylabel("Preço (USDT)")
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## Resumo

Este notebook demonstrou:

1. **Carregamento de dados reais** em formato Parquet
2. **Multi-asset analysis** (BTC, ETH, ADA, etc.)
3. **Indicadores técnicos** (SMA, volatilidade)
4. **Feature engineering** para ML
5. **Arquitetura Medallion** (Bronze → Silver → Gold)

Próximos passos para o projeto:
- Aplicar algoritmo de clustering (K-Means) para regimes de mercado
- Implementar backtesting de estratégias
- Usar Structured Streaming para dados em tempo real